<a href="https://www.kaggle.com/code/rudrajit55555/eda-rudrajitdas?scriptVersionId=357089227" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/satyakidas07/retail-sales-dataset/retail_sales_dataset.csv


# 1. Importing Warnings

In [2]:
# Suppress warnings for a cleaner notebook output
import warnings
warnings.filterwarnings('ignore')

# 2. Importing Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
from sklearn.model_selection import train_test_split #split data into training and testing sets
from sklearn.model_selection import GridSearchCV #Exhaustive Search
from sklearn.linear_model import LinearRegression #Standard Linear Modelling
from sklearn.metrics import mean_squared_error, r2_score #For Regrssion performance
from sklearn.preprocessing import StandardScaler #standardize features (mean=0, variance=1)
from sklearn.preprocessing import MinMaxScaler #scale features to a range 


# 3. Load Dataset

In [4]:
df = pd.read_csv("/kaggle/input/datasets/satyakidas07/retail-sales-dataset/retail_sales_dataset.csv")
df.head()

,order_id,order_date,customer_id,customer_name,age,gender,region,city,product_category,product_name,...,unit_price,discount_pct,sales_amount,profit,shipping_cost,payment_method,customer_satisfaction,return_flag,order_status,days_to_ship
0,ORD-03903,2024-08-15,CUST32603,Aarav Mehta,39.0,Other,Central,Raipur,Groceries,Sugar,...,346.32,NaN,4043.14,289.18,56.04,Debit Card,NaN,True,Returned,7.0
1,ORD-02198,2022-08-16,CUST07881,Sanjay Verma,28.0,Male,West,Mumbai,Beauty,Sunscreen,...,858.66,0.08,9147.31,5355.56,64.49,Debit Card,5.0,False,Delivered,2.0
2,ORD-01348,2021-08-29,CUST49296,Pooja Sharma,41.0,Other,West,Surat,Furniture,Sofa,...,25941.32,0.14,110156.97,36956.51,67.65,Debit Card,4.0,False,Delivered,5.0
3,ORD-02076,2022-07-05,CUST05587,Priya Reddy,29.0,Other,West,Ahmedabad,Furniture,Bed Frame,...,22071.92,0.17,262066.11,66644.99,54.47,EMI,3.0,False,Delivered,4.0
4,ORD-00287,2020-05-07,CUST58035,Ritu Sharma,9.0,Male,Central,Nagpur,Groceries,Cooking Oil,...,320.06,0.31,323.95,31.38,48.81,Net Banking,4.0,False,Pending,4.0


# 4. Inspect Raw Dataset


**Total Count of Rows & Columns**

In [5]:
n_rows = df.shape[0]
n_cols = df.shape[1]
print(f"Total Rows: {n_rows}")
print(f"Total Columns: {n_cols}")

Total Rows: 4310
Total Columns: 21


In [6]:
df.info # Full info of dataset

<bound method DataFrame.info of        order_id  order_date customer_id  customer_name   age  gender   region  \
0     ORD-03903  2024-08-15   CUST32603    Aarav Mehta  39.0   Other  Central   
1     ORD-02198  2022-08-16   CUST07881   Sanjay Verma  28.0    Male     West   
2     ORD-01348  2021-08-29   CUST49296   Pooja Sharma  41.0   Other     West   
3     ORD-02076  2022-07-05   CUST05587    Priya Reddy  29.0   Other     West   
4     ORD-00287  2020-05-07   CUST58035    Ritu Sharma   9.0    Male  Central   
...         ...         ...         ...            ...   ...     ...      ...   
4305  ORD-00056  2020-01-21   CUST85136  Sanjay Sharma  45.0       F     West   
4306  ORD-03458  2024-02-08   CUST09746   Karan Pillai  25.0       m    South   
4307  ORD-01769  2022-02-22   CUST64190    Karan Reddy  39.0    Male     East   
4308  ORD-01738  2022-02-11   CUST50460     Amit Mehta  40.0  Female     East   
4309  ORD-03241  2023-11-01   CUST36164     Nikhil Rao  23.0   Other    North

***Required informations from the dataset***

In [7]:
info_df = pd.DataFrame({
    "dtype": df.dtypes.astype(str), # data types 
    "unique_values": df.nunique(), # total number of unique entries in a column
    "sample_value": df.iloc[1], # example of entries in the columns
    "missing_values": df.isnull().sum(), # checking null_values
    "Total memory in mb": df.memory_usage(deep=True)/(1024 ** 2) # memory usage by each column
})
info_df

,dtype,unique_values,sample_value,missing_values,Total memory in mb
Index,NaN,NaN,NaN,NaN,0.000126
age,float64,80.0,28.0,160.0,0.032883
city,object,20.0,Mumbai,30.0,0.229590
customer_id,object,4112.0,CUST07881,30.0,0.237656
customer_name,object,400.0,Sanjay Verma,30.0,0.247028
customer_satisfaction,float64,5.0,5.0,378.0,0.032883
days_to_ship,float64,14.0,2.0,130.0,0.032883
discount_pct,float64,41.0,0.08,167.0,0.032883
gender,object,11.0,Male,30.0,0.220455
order_date,object,1837.0,2022-08-16,30.0,0.242325


**Statistical Summary**

In [8]:
df.describe()

,age,quantity,unit_price,discount_pct,sales_amount,profit,shipping_cost,customer_satisfaction,days_to_ship
count,4150.000000,4170.000000,4280.000000,4143.000000,4.280000e+03,4.280000e+03,4280.000000,3932.000000,4180.000000
mean,35.335181,7.379856,10621.154428,0.202085,6.235864e+04,1.168713e+04,63.358836,3.016531,5.620335
std,35.709382,43.578352,17323.704016,0.118095,3.166998e+05,5.287522e+04,18.312281,1.413127,4.178663
min,-7.000000,-1.000000,50.260000,0.000000,3.945000e+01,1.560000e+00,-4.290000,1.000000,-1.000000
25%,26.000000,3.000000,672.222500,0.100000,2.462165e+03,7.018225e+02,50.817500,2.000000,3.000000
50%,34.000000,5.000000,2228.435000,0.210000,9.548075e+03,3.128005e+03,63.465000,3.000000,6.000000
75%,42.000000,8.000000,12717.435000,0.310000,5.006964e+04,1.040372e+04,75.650000,4.000000,8.000000
max,999.000000,999.000000,79957.920000,0.400000,1.736031e+07,2.445648e+06,133.350000,5.000000,100.000000


In [9]:
# check duplicate values
full_dup = df.duplicated().sum()
dup_order_id = df["order_id"].duplicated().sum()
print(f"Full duplicated rows: {full_dup}")
print(f"Duplicated order_id : {dup_order_id}")

Full duplicated rows: 109
Duplicated order_id : 109


# Dataset Cleaning

In [10]:
# Drop unnecessary column
df = df.drop(columns = ['customer_name'])

In [11]:
# removing blank rows
df.dropna(how='all', inplace=True)

In [12]:
df.isnull().sum()

order_id                   0
order_date                 0
customer_id                0
age                      130
gender                     0
region                     0
city                       0
product_category           0
product_name               0
quantity                 110
unit_price                 0
discount_pct             137
sales_amount               0
profit                     0
shipping_cost              0
payment_method             0
customer_satisfaction    348
return_flag                0
order_status               0
days_to_ship             100
dtype: int64

In [13]:
# Removing duplicate entries

df.loc[df.duplicated(subset='order_id', keep=False)].sort_values("order_id").head(10)

,order_id,order_date,customer_id,age,gender,region,city,product_category,product_name,quantity,unit_price,discount_pct,sales_amount,profit,shipping_cost,payment_method,customer_satisfaction,return_flag,order_status,days_to_ship
590,ORD-00051,2020-01-20,CUST27973,13.0,Male,South,Hyderabad,Beauty,Perfume,6.0,1952.59,0.14,6658.21,3867.94,38.17,Credit Card,1.0,False,Shipped,8.0
3899,ORD-00051,2020-01-20,CUST27973,13.0,Male,South,Hyderabad,Beauty,Perfume,6.0,1952.59,0.14,6658.21,3867.94,38.17,Credit Card,1.0,False,Shipped,8.0
1654,ORD-00084,2020-02-03,CUST43984,58.0,Male,North,Ludhiana,Clothing,Kurta,2.0,1849.33,0.36,2115.12,453.68,31.92,UPI,5.0,False,Delivered,5.0
1111,ORD-00084,2020-02-03,CUST43984,58.0,Male,North,Ludhiana,Clothing,Kurta,2.0,1849.33,0.36,2115.12,453.68,31.92,UPI,5.0,False,Delivered,5.0
738,ORD-00144,2020-02-27,CUST57714,37.0,Female,South,Chennai,Furniture,Chair,5.0,32167.65,NaN,138458.15,39811.27,58.32,Cash on Delivery,5.0,False,Delivered,6.0
1957,ORD-00144,2020-02-27,CUST57714,37.0,Female,South,Chennai,Furniture,Chair,5.0,32167.65,NaN,138458.15,39811.27,58.32,Cash on Delivery,5.0,False,Delivered,6.0
128,ORD-00268,2020-04-27,CUST64785,32.0,Female,West,Mumbai,Books,Cook Book,7.0,257.18,0.29,1452.88,604.81,60.36,Credit Card,1.0,False,Cancelled,5.0
774,ORD-00268,2020-04-27,CUST64785,32.0,Female,West,Mumbai,Books,Cook Book,7.0,257.18,0.29,1452.88,604.81,60.36,Credit Card,1.0,False,Cancelled,5.0
941,ORD-00305,2020-05-12,CUST10300,35.0,Other,East,Bhubaneswar,Sports,Yoga Mat,5.0,6852.57,0.19,30208.26,8215.56,100.86,EMI,5.0,False,Delivered,10.0
715,ORD-00305,2020-05-12,CUST10300,35.0,Other,East,Bhubaneswar,Sports,Yoga Mat,5.0,6852.57,0.19,30208.26,8215.56,100.86,EMI,5.0,False,Delivered,10.0


In [14]:
# remove 'ORD' from the column order_id
try:
  df['order_id'] = df['order_id'].str.replace("ORD-0","",regex=False)
  display(df.head())

except:
  print("Removed")

,order_id,order_date,customer_id,age,gender,region,city,product_category,product_name,quantity,unit_price,discount_pct,sales_amount,profit,shipping_cost,payment_method,customer_satisfaction,return_flag,order_status,days_to_ship
0,3903,2024-08-15,CUST32603,39.0,Other,Central,Raipur,Groceries,Sugar,8.0,346.32,NaN,4043.14,289.18,56.04,Debit Card,NaN,True,Returned,7.0
1,2198,2022-08-16,CUST07881,28.0,Male,West,Mumbai,Beauty,Sunscreen,8.0,858.66,0.08,9147.31,5355.56,64.49,Debit Card,5.0,False,Delivered,2.0
2,1348,2021-08-29,CUST49296,41.0,Other,West,Surat,Furniture,Sofa,4.0,25941.32,0.14,110156.97,36956.51,67.65,Debit Card,4.0,False,Delivered,5.0
3,2076,2022-07-05,CUST05587,29.0,Other,West,Ahmedabad,Furniture,Bed Frame,9.0,22071.92,0.17,262066.11,66644.99,54.47,EMI,3.0,False,Delivered,4.0
4,0287,2020-05-07,CUST58035,9.0,Male,Central,Nagpur,Groceries,Cooking Oil,1.0,320.06,0.31,323.95,31.38,48.81,Net Banking,4.0,False,Pending,4.0


In [15]:
# removing null_values from order_id
df = df.dropna(subset= ['order_id'])
df[df['order_id'].isna()]

,order_id,order_date,customer_id,age,gender,region,city,product_category,product_name,quantity,unit_price,discount_pct,sales_amount,profit,shipping_cost,payment_method,customer_satisfaction,return_flag,order_status,days_to_ship


In [16]:
# change datatype of order_id column
df['order_id'] = df['order_id'].astype(int)
df['order_id'].dtypes

dtype('int64')

In [17]:
df['age'].unique()

array([ 39.,  28.,  41.,  29.,   9.,  31.,  43.,  44.,  17.,  51.,  14.,
        15.,  53.,  30.,  26.,  21.,  35.,  22.,  20.,  65.,  27.,  52.,
        42.,  40.,  -6.,  49., 999.,  34.,  38.,  57.,  36.,  48.,  37.,
        nan,  32.,  33.,  45.,  23.,  16.,  10.,  25.,  24.,  11.,  12.,
        50.,  54.,  46.,  56.,  47.,  19.,  18.,  58.,  -5.,  66.,   8.,
        61.,  55.,   5.,   2.,  60.,  63.,  13.,  59.,   0.,   6.,  -1.,
        -2.,   7., 150.,  62.,  69.,   4.,  68.,   3.,  71.,  -4.,  64.,
         1.,  67.,  72.,  -7.])

In [18]:
# negative age to be replaced as NaN
df.loc[(df['age'] <= 0) | (df['age']>120), 'age'] = np.nan
df['age'].unique()

array([39., 28., 41., 29.,  9., 31., 43., 44., 17., 51., 14., 15., 53.,
       30., 26., 21., 35., 22., 20., 65., 27., 52., 42., 40., nan, 49.,
       34., 38., 57., 36., 48., 37., 32., 33., 45., 23., 16., 10., 25.,
       24., 11., 12., 50., 54., 46., 56., 47., 19., 18., 58., 66.,  8.,
       61., 55.,  5.,  2., 60., 63., 13., 59.,  6.,  7., 62., 69.,  4.,
       68.,  3., 71., 64.,  1., 67., 72.])

In [19]:
# filling null values in age column with median age
med_age = df['age'].median()
df['age'].fillna(med_age,inplace=True)
df['age'].isnull().sum()

np.int64(0)

In [20]:
df['quantity'].unique()

array([  8.,   4.,   9.,   1.,   2.,   3.,   7.,  10.,   5.,   6.,  nan,
        -1.,   0., 999.])

In [21]:
# negative order quantity to be replaced as NaN
df.loc[df['quantity'] <= 0, 'quantity'] = np.nan
df['quantity'].unique()

array([  8.,   4.,   9.,   1.,   2.,   3.,   7.,  10.,   5.,   6.,  nan,
       999.])

In [22]:
# filling null values in quantity column with median
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['quantity'] = df['quantity'].astype(int)
df['quantity'].dtypes
df['quantity'].isnull().sum()

np.int64(0)

In [23]:
df['quantity'].dtype

dtype('int64')

In [24]:
# filling null values in discount_pct column with median
df['discount_pct'].median()
df['discount_pct'] = df['discount_pct'].fillna(df['discount_pct'].median())
df['discount_pct'].isnull().sum()

np.int64(0)

In [25]:
df['customer_satisfaction'].unique()

array([nan,  5.,  4.,  3.,  2.,  1.])

In [26]:
# replacing NaN values in customer_satisfaction column with median
df['customer_satisfaction'].median()
df['customer_satisfaction'] = df['customer_satisfaction'].fillna(df['customer_satisfaction'].median())
df['customer_satisfaction'].isnull().sum()

np.int64(0)

In [27]:
df['days_to_ship'].unique()

array([  7.,   2.,   5.,   4.,   8.,  10.,   9.,  nan,   1.,   6.,   3.,
        45.,   0.,  -1., 100.])

In [28]:
# days_to_ship should be positive, thus replacing negetive values with NaN
df.loc[df['days_to_ship'] <= 0, 'days_to_ship'] = np.nan
df['days_to_ship'].unique()

array([  7.,   2.,   5.,   4.,   8.,  10.,   9.,  nan,   1.,   6.,   3.,
        45., 100.])

In [29]:
# replacing NaN values in days_to_ship column with median
df['days_to_ship'].median()
df['days_to_ship'] = df['days_to_ship'].fillna(df['days_to_ship'].median())
df['days_to_ship'].isnull().sum()

np.int64(0)

In [30]:
df['days_to_ship'] = df['days_to_ship'].astype('int64')
df['days_to_ship'].dtypes

dtype('int64')

In [31]:
df['order_date'].unique()

array(['2024-08-15', '2022-08-16', '2021-08-29', ..., '2020-12-16',
       '16/09/2023', '2022-02-11'], dtype=object)

In [32]:
# fix order_date format
df['order_date'] = pd.to_datetime(df['order_date'],errors= 'coerce').dt.strftime('%Y/%m/%d')
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_date'].dtypes

dtype('<M8[ns]')

In [33]:
df.isnull().sum()

order_id                   0
order_date               273
customer_id                0
age                        0
gender                     0
region                     0
city                       0
product_category           0
product_name               0
quantity                   0
unit_price                 0
discount_pct               0
sales_amount               0
profit                     0
shipping_cost              0
payment_method             0
customer_satisfaction      0
return_flag                0
order_status               0
days_to_ship               0
dtype: int64

In [34]:
columns = list(df.columns)
#print(columns)
for i in columns:
  print(i,end = '')
  print(df[i].unique())

order_id[3903 2198 1348 ... 1769 1738 3241]
order_date<DatetimeArray>
['2024-08-15 00:00:00', '2022-08-16 00:00:00', '2021-08-29 00:00:00',
 '2022-07-05 00:00:00', '2020-05-07 00:00:00',                 'NaT',
 '2021-09-28 00:00:00', '2023-06-10 00:00:00', '2020-07-24 00:00:00',
 '2020-02-09 00:00:00',
 ...
 '2023-07-13 00:00:00', '2022-04-13 00:00:00', '2022-11-03 00:00:00',
 '2020-05-18 00:00:00', '2024-07-16 00:00:00', '2024-04-12 00:00:00',
 '2020-10-08 00:00:00', '2020-01-04 00:00:00', '2020-12-16 00:00:00',
 '2022-02-11 00:00:00']
Length: 1573, dtype: datetime64[ns]
customer_id['CUST32603' 'CUST07881' 'CUST49296' ... 'CUST64190' 'CUST50460'
 'CUST36164']
age[39. 28. 41. 29.  9. 31. 43. 44. 17. 51. 14. 15. 53. 30. 26. 21. 35. 22.
 20. 65. 27. 52. 42. 40. 34. 49. 38. 57. 36. 48. 37. 32. 33. 45. 23. 16.
 10. 25. 24. 11. 12. 50. 54. 46. 56. 47. 19. 18. 58. 66.  8. 61. 55.  5.
  2. 60. 63. 13. 59.  6.  7. 62. 69.  4. 68.  3. 71. 64.  1. 67. 72.]
gender['Other' 'Male' 'Female' 'FEMALE'